# 08Y · Which genes track T-cell lineage within mice?

**Adagrasib mouse Visium HD · Gene discovery**  
Unzip the package under the CT26 project used for 08V. Open this notebook with the project's Python kernel and run the cells in order. If the working directory is elsewhere, point `PROJECT_ROOT` below to the CT26 project. The companion notebook, 08Z, can run in a fresh kernel.

Each point on the map is a gene. The signed associations estimate whether a gene and an anchor are detected in the **same segmented cell**, within a mouse and at similar library depth. Existing 50 µm neighborhood measurements provide a second way to assess possible environmental contributions.

## Inputs and discovery settings

For the UMAP and Leiden outputs, the kernel needs `umap-learn`, `igraph`, and `leidenalg`. If any is missing, install it in this kernel, then restart and run from the top. A kernel without these packages labels the fallback PCA projection or Ward clustering in the plot and run settings.

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd
from IPython.display import display, Image, Markdown
from input08y import open_bundle
from workflow08yz import marker_audit, source_arguments, start_discovery

# Leave None when this package is inside the CT26 project that ran 08V.
# From another directory, point PROJECT_ROOT to that project or the CT26 run root.
PROJECT_ROOT = None
OUTPUT_ROOT = Path(os.getenv("CT26_08YZ_OUTPUT", "08YZ_results")).expanduser().resolve()

# Optional explicit sources. An alternate H5AD must already contain only the
# selected T cells, all measured genes, and observed counts in layers["counts"].
COUNTS_H5AD = None
DESIGN_CSV = None
MIXED_CSV = None
RATIO_BY_MOUSE_CSV = None

source = source_arguments(run_root=PROJECT_ROOT, counts_h5ad=COUNTS_H5AD,
                          design_csv=DESIGN_CSV, mixed_csv=MIXED_CSV,
                          ratio_csv=RATIO_BY_MOUSE_CSV)
# Advanced example for a different H5AD storing raw counts in X:
# source.update(layer="X", sample_col="sample_id", gene_col="gene_symbol")

DISCOVERY = dict(seed=17, max_cells_per_mouse=30_000, n_depth_bins=10,
                 min_detection=12, min_agreement=0.75, min_r=0.015,
                 min_margin=0.01, max_signature=16, max_env_ratio=1.0,
                 max_map_genes=8_000, sketch_dim=192, n_pcs=35, map_neighbors=20,
                 resolution=1.0, use_umap=True, use_leiden=True)
print("Result directory:", OUTPUT_ROOT)

## Verify the cohort before fitting

For the reviewed CT26 run, expect 590,247 cells from 12 mice, including 486,130 cells with no observed coreceptor. The four 08X evaluation mice are Ada-1, Ada-12, Ada-15 and Ada-6. Check the mixed-flag and ratio coverage printed here. Unknown mixed status is retained explicitly.

In [ ]:
data = open_bundle(**source)
X, obs, genes, gmeta = data
audit = marker_audit(X, obs, genes)
display(audit)
print("Cells with neither coreceptor:", f"{audit.neither.sum():,}")
print("Known mixed status:", f"{obs.mixed.notna().sum():,} / {len(obs):,}")
print("Discovery-only and pooled environmental ratio coverage:")
display(gmeta[["env_ratio", "env_ratio_pooled"]].notna().sum().rename("genes"))

## Fit the gene map and freeze candidate signatures

Detection is adjusted using depth strata within each discovery mouse. The notebook computes signed anchor associations directly and uses a compressed representation of the depth-adjusted gene profiles for PCA, nearest neighbors, UMAP, and module detection. The graph includes informative lineage, state, and non-T marker genes. Feature selection also considers replication across mice, agreement across conditions, the opposing lineage, and associations with state and non-T control programs.

Where the discovery mice have 50 µm measurements, the ratio of neighboring non-T expression to focal T-cell expression contributes to candidate selection. Pooled ratios are available for interpretation. `max_cells_per_mouse` limits the discovery calculation. 08Z scores every cell.

In [ ]:
options = dict(DISCOVERY, depth_col="total_counts",
               ratio_scope="discovery" if gmeta.env_ratio.notna().any() else "descriptive")
ydir = start_discovery(OUTPUT_ROOT, data, source, options)
settings = json.loads((ydir / "discovery_settings.json").read_text())
signatures = json.loads((ydir / "signatures.json").read_text())
print("Completed discovery:", ydir)
print("Embedding:", settings["embedding"], "| Modules:", settings["clustering"])
display(pd.DataFrame([{"lineage": key, "n_genes": len(panel), "genes": ", ".join(panel)}
                      for key, panel in signatures.items()]))
display(Image(filename=str(ydir / "gene_map.png"), width=850))
evidence = ydir / "gene_map_evidence.png"
if evidence.is_file():
    display(Image(filename=str(evidence), width=950))

## Inspect individual anchor neighbors

Change `ANCHORS` to explore e.g. `Cd40lg`, `Zbtb7b`, `cytotoxic`, `regulatory`, `myeloid` or `epithelial`. The table ranks genes by signed association across mice. The second table explains each candidate's selection or exclusion. Two-dimensional distance is only a display aid.

In [ ]:
ANCHORS = ["Cd4", "Cd8a", "Cd8b1", "Foxp3"]
rankings = pd.read_csv(ydir / "anchor_rankings.csv.gz")
shown = rankings.loc[rankings.anchor.isin(ANCHORS) & rankings.neighbor_rank.notna()]
display(shown.sort_values(["anchor", "neighbor_rank"]).groupby("anchor", sort=False).head(8)
        [["anchor", "gene", "neighbor_rank", "mouse_replicated_r", "mice_with_r", "mice_positive", "env_ratio"]])
candidates = pd.read_csv(ydir / "lineage_candidate_audit.csv.gz")
display(candidates.sort_values(["lineage", "selected", "median_paired_r_difference"],
                               ascending=[True, False, False]).groupby("lineage").head(10)
        [["lineage", "gene", "selected", "median_target_r", "median_opposite_r",
          "closest_state_control", "closest_nonT_control", "env_ratio", "exclusion_reasons"]])
print("08Z will reopen this run from:", OUTPUT_ROOT / "latest_08Y.json")

**Interpretation.** A recurring association can reflect lineage, activation, spatial context, or RNA assigned across a segmentation boundary. An empty candidate signature is a valid result. Open 08Z to test masked-coreceptor recovery in the four evaluation mice and obtain cell-level state scores.

## Download the return-to-chat ZIP

This cell can run on its own after a completed analysis. If you used a custom output directory, set `REVIEW_ROOT` below. The ZIP appears beside this notebook and includes an HTML figure index and aggregate results.

In [ ]:
from pathlib import Path
import os
from IPython.display import FileLink, display
from review08yz import return_zip

REVIEW_ROOT = globals().get("OUTPUT_ROOT", Path(os.getenv("CT26_08YZ_OUTPUT", "08YZ_results")))
archive = return_zip(REVIEW_ROOT, stage="08Y", ydir=globals().get("ydir"))
print("Return ZIP:", archive)
display(FileLink(archive.name))